# EDA: Provider Log

## Setup
Reset database and seed base data.

In [1]:
from pathlib import Path
import os, sys
from dotenv import load_dotenv

# Set working directory explicitly to project root
PROJECT_ROOT = Path(r"C:\Repos\codecritic").resolve()

os.chdir(PROJECT_ROOT)  # Change current directory
sys.path.insert(0, str(PROJECT_ROOT))  # Ensure imports resolve correctly

print("Working directory is now:", Path.cwd())

from app.db.connection import DB_PATH, close_connection
from sqlalchemy import create_engine
from app.db.seeders.seed_score_providers import seed_score_providers
from app.db.seeders.seed_tool_providers import seed_tool_providers
from app.db.seeders.seed_prompt_providers import seed_prompt_providers
from app.db.seeders.seed_prompts import seed_prompts
from app.db.seeders.seed_context_providers import seed_context_providers
from app.db.seeders.seed_agent_engine_providers import seed_agent_engine_providers
from app.db.seeders.seed_agent_providers import seed_agent_providers
from app.db.seeders.seed_state_providers import seed_state_providers
from app.db.seeders.seed_system_providers import seed_system_providers
from app.db.seeders.seed_controller_providers import seed_controller_providers
from app.db.seeders.seed_program_provider import seed_program_providers
from app.db.seeders.seed_session_configs import seed_session_configs
from app.db.base import Base
from sqlalchemy.orm import Session

# 🧹 Reset DB
close_connection()
engine = create_engine(f"sqlite:///{DB_PATH}")
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

# 🌱 Seed database records
with Session(bind=engine) as session:
    seed_prompts(session)
    seed_prompt_providers(session)
    seed_tool_providers(session)
    seed_score_providers(session)
    seed_context_providers(session)
    seed_agent_engine_providers(session)
    seed_agent_providers(session)
    seed_state_providers(session)
    seed_system_providers(session)
    seed_controller_providers(session)
    seed_program_providers(session)
    seed_session_configs(session)

# Load from env/.env relative to project root
dotenv_path = Path("env/.env").resolve()
if dotenv_path.exists():
    load_dotenv(dotenv_path)
    print("✅ Loaded environment variables from env/.env")
else:
    raise FileNotFoundError(f"❌ Missing .env file at: {dotenv_path}")


Working directory is now: C:\Repos\codecritic
Seeded AgentPrompt 'generate' with ID: 1 and GUID: 7e564100-5e94-43d0-8c65-64214817e8ba
Seeded AgentPrompt 'linting_generator_agent' with ID: 2 and GUID: f1d8bdff-68b9-425e-86c5-d83afaafa750
Seeded SystemPrompt 'format' with ID: 1 and GUID: 64054c6d-9ec9-4c53-b42f-fde3836cbe1e
Seeded SystemPrompt 'linting_system' with ID: 2 and GUID: e998bea7-a8f9-4e4d-9e2a-45aff1680cd2
Seeded prompt provider configurations successfully.
Seeded tool configurations successfully.
Seeded score providers successfully.
Seeded context provider configurations successfully.
Seeded agent engine configurations successfully.
Seeded agent provider configurations successfully.
✅ Seeded state provider configurations successfully.
Seeded system provider configurations successfully.
✅ Seeded controller providers
✅ Seeded program providers
✅ Seeded session configs
✅ Loaded environment variables from env/.env


In [2]:
# 🧪 Run Program Using SessionConfig

from app.db.models import SessionConfig
from app.enums.logging_enums import PROVIDER_TYPE
from app.enums.system_enums import SYSTEM_TYPE
from app.factories.program_provider_factory import ProgramProviderFactory
from sqlalchemy.orm import Session
from pathlib import Path
import json

# Fetch session config
with Session(bind=engine) as db:
    session_row = db.query(SessionConfig).filter_by(id=1).first()
    assert session_row, "❌ No session config found"

# Prepare test file
file = Path("tests/session_linked_execution.py")
file.write_text("def ping( user ):\n return f\"pong {user}\"")

# Construct input using session metadata
input_data = {
    "file_path":  str(file),
    "session_id": session_row.id,
    "system":     SYSTEM_TYPE.LINTING.value
}

# Run associated program
program = ProgramProviderFactory.create(
    id=session_row.program_provider_id,
    called_by_type=PROVIDER_TYPE.SESSION,
    called_by_id=session_row.id
)
result = program.run(input_data, session_id=session_row.id)

print("✅ Final Program Result (via SessionConfig):")
print(json.dumps(result.model_dump(), indent=2))



PROGRAM TEST
file1  working_files\final_0938352895.py
file2  tests\session_linked_execution.py
BEST FILE:  C:\Repos\codecritic\working_files\final_0938352895.py

✅ Final Program Result (via SessionConfig):
{
  "state": "end",
  "previous_state": "preprocessing",
  "state_type": "end",
  "decision": "accept",
  "steps": 2,
  "max_steps": 20,
  "summary": "preprocessing complete",
  "output": {
    "state": "end",
    "file_path": "working_files\\final_0938367774.py",
    "session_id": 1,
    "system": "linting",
    "reason": "preprocessing complete",
    "steps": 2,
    "retry_count": 0,
    "_last_state": "preprocessing",
    "decision": "accept",
    "score": null,
    "state_output": {
      "state": "end",
      "previous_state": "preprocess",
      "state_type": "end",
      "decision": "accept",
      "steps": 2,
      "max_steps": 10,
      "summary": "Completed",
      "provider_name": "preprocessing_controller"
    }
  },
  "provider_name": "codecritic_program"
}


### Load raw log entries

In [3]:
import sqlite3
import pandas as pd
import json
from app.db.connection import DB_PATH

# Connect and query
conn = sqlite3.connect(DB_PATH)
df = pd.read_sql("SELECT * FROM provider_log WHERE session_id='1'", conn)
conn.close()

# Pretty-print each log entry
for i, row in df.iterrows():
    print(f"\n🔹 Entry {i + 1} — ID {row['id']}")
    print(f"🕒 Timestamp: {row['timestamp']}")
    print(f"🧩 Provider ID: {row['provider_id']} ({row['provider_type']})")
    print(f"📦 Output Schema: {row['output_schema']}")
    print(f"⏱️ Latency: {row['latency_ms']} ms")
    print(f"📄 File Name: {row['file_name']}")
    print(f"🔑 Config Hash: {row['config_hash']}")
    
    # 🔁 Caller trace
    called_by_type = row.get("called_by_type", None)
    called_by_id = row.get("called_by_id", None)
    if called_by_type or called_by_id:
        print(f"🧭 Called By: {called_by_type} (ID {called_by_id})")

    # Safely parse JSON input/output if possible
    try:
        parsed_input = json.loads(row['input'])
        parsed_output = json.loads(row['output'])
        print("📥 Input:")
        print(json.dumps(parsed_input, indent=2))
        print("📤 Output:")
        print(json.dumps(parsed_output, indent=2))
    except Exception:
        print(f"📥 Input (raw): {row['input']}")
        print(f"📤 Output (raw): {row['output']}")

    print("—" * 80)



🔹 Entry 1 — ID 1
🕒 Timestamp: 2025-06-07T14:38:21.315559+00:00
🧩 Provider ID: 2 (score)
📦 Output Schema: ScoreOutputSchema
⏱️ Latency: 2 ms
📄 File Name: 0a6d31ed-fe88-4a62-99cd-b3cd139d7c8d.py
🔑 Config Hash: 1b647994448738282cd3340596eed1f8
🧭 Called By: agent (ID 4.0)
📥 Input:
{
  "state": "code_stability",
  "file_path": "C:\\Repos\\codecritic\\working_files\\session_linked_execution.__state_0938213125.py",
  "session_id": 1,
  "system": "linting",
  "reason": "entering stability check",
  "steps": 3,
  "retry_count": 0,
  "_last_state": "start",
  "decision": "unknown"
}
📤 Output:
{
  "name": "code_stability_score",
  "value": 1.0,
  "components": "{\"utf8_valid\": 1.0, \"syntax_ok\": 1.0, \"can_compile\": 1.0, \"py_compile_ok\": 1.0, \"can_import\": 1.0}",
  "summary": "\u2705 Executable"
}
————————————————————————————————————————————————————————————————————————————————

🔹 Entry 2 — ID 2
🕒 Timestamp: 2025-06-07T14:38:21.315559+00:00
🧩 Provider ID: 4 (agent)
📦 Output Schema: AgentOu